# One Kaggle T4x2 Experiment - Direct Dataset

Runs one YOLO11s segmentation experiment using the Kaggle dataset `ayushbpanchal/indraeye-seg` directly. No large project ZIP upload is needed. Start with `RUN_STAGE = "smoke"`; after it passes, change only `RUN_STAGE` to `"full"`.

In [ ]:
from pathlib import Path

RUN_STAGE = "smoke"  # "smoke" first, then "full"

GITHUB_REPO = "https://github.com/AyushPanchal/domain-adaptation-segmentation.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "domain-adaptation-segmentation"
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/ayushbpanchal/indraeye-seg"),
    Path("/kaggle/input/indraeye-seg"),
]
DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), DATASET_ROOT_CANDIDATES[0])

EXPERIMENT_NAME = "E01_source_rgb_yolo11s"
DATASET_YAML = "data/manifests/dataset_yamls/kaggle_direct_source_rgb.yaml"
EVAL_IR_YAML = "data/manifests/dataset_yamls/kaggle_direct_eval_ir.yaml"
EVAL_EO_IR_YAML = "data/manifests/dataset_yamls/kaggle_direct_eval_eo_ir.yaml"
EXPERIMENT_CONFIG = "configs/experiments/e01_kaggle_direct_source_rgb_yolo11s.yaml"

YOLO_DEVICE = "0,1"      # Kaggle T4x2 training
YOLO_EVAL_DEVICE = "0"    # evaluation is cheap; single GPU avoids DDP val quirks
YOLO_BATCH = "16"        # split across both GPUs during training
YOLO_WORKERS = "2"
YOLO_PATIENCE = "25"
YOLO_RESUME = "auto"
YOLO_EPOCHS = "1" if RUN_STAGE == "smoke" else "100"

OUTPUT_ROOT = WORK_DIR / "runs" / f"kaggle_direct_e01_{RUN_STAGE}"
REPORT_DIR = Path(f"reports/tables/kaggle_direct_e01_{RUN_STAGE}")

assert RUN_STAGE in {"smoke", "full"}, RUN_STAGE
print("RUN_STAGE:", RUN_STAGE)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("YOLO_DEVICE/BATCH/EPOCHS:", YOLO_DEVICE, YOLO_BATCH, YOLO_EPOCHS)


## Helpers

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime

def stage(title):
    print("\n" + "=" * 92)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {title}")
    print("=" * 92)

def run_live(command, cwd=None, env=None):
    stage("RUN: " + " ".join(map(str, command)))
    process = subprocess.Popen(
        list(map(str, command)), cwd=str(cwd) if cwd else None, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    print(f"\n[return_code] {return_code}")
    if return_code != 0:
        raise RuntimeError(f"Command failed with return code {return_code}: {command}")

def show_tail(path, lines=60):
    path = Path(path)
    stage(f"TAIL: {path}")
    if not path.exists():
        print("missing")
        return
    text = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(text[-lines:]))

def count_files(path, suffixes):
    path = Path(path)
    return sum(1 for item in path.rglob("*") if item.suffix.lower() in suffixes)

def running_on_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


## Stage 1 - Clone Code Repo

In [ ]:
stage("Stage 1 - Clone or update code repo")
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run_live(["git", "pull"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
else:
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])

print("Repo:", REPO_DIR)
print("Commit:")
run_live(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)


## Stage 2 - Verify Kaggle Dataset And Create YAMLs

In [ ]:
stage("Stage 2 - Verify Kaggle dataset")
required_dirs = [
    DATASET_ROOT / "eo/images/train",
    DATASET_ROOT / "eo/labels/train",
    DATASET_ROOT / "eo/images/val",
    DATASET_ROOT / "eo/labels/val",
    DATASET_ROOT / "ir/images/val",
    DATASET_ROOT / "ir/labels/val",
]
for path in required_dirs:
    print(path, "OK" if path.exists() else "MISSING")
    if not path.exists():
        raise FileNotFoundError(path)

print("EO train images:", count_files(DATASET_ROOT / "eo/images/train", {".jpg", ".jpeg", ".png"}))
print("EO train labels:", count_files(DATASET_ROOT / "eo/labels/train", {".txt"}))
print("EO val images:", count_files(DATASET_ROOT / "eo/images/val", {".jpg", ".jpeg", ".png"}))
print("EO val labels:", count_files(DATASET_ROOT / "eo/labels/val", {".txt"}))
print("IR val images:", count_files(DATASET_ROOT / "ir/images/val", {".jpg", ".jpeg", ".png"}))
print("IR val labels:", count_files(DATASET_ROOT / "ir/labels/val", {".txt"}))

stage("Stage 2 - Write train and evaluation YAMLs")

def class_block():
    return """nc: 12
names:
  0: Bicycle
  1: Bus
  2: Car
  3: Cargo trike
  4: Ignore
  5: Motorcycle
  6: Person
  7: Rickshaw
  8: Small truck
  9: Tractor
  10: Truck
  11: Van
"""

dataset_yaml_path = REPO_DIR / DATASET_YAML
eval_ir_yaml_path = REPO_DIR / EVAL_IR_YAML
eval_eo_ir_yaml_path = REPO_DIR / EVAL_EO_IR_YAML
for yaml_path in [dataset_yaml_path, eval_ir_yaml_path, eval_eo_ir_yaml_path]:
    yaml_path.parent.mkdir(parents=True, exist_ok=True)

# Training uses EO train and IR validation for early stopping/domain-transfer baseline.
dataset_yaml_path.write_text(f"""path: {DATASET_ROOT.as_posix()}
train: eo/images/train
val: ir/images/val

{class_block()}""", encoding="utf-8")

# Explicit final evaluation 1: IR-only.
eval_ir_yaml_path.write_text(f"""path: {DATASET_ROOT.as_posix()}
train: eo/images/train
val: ir/images/val

{class_block()}""", encoding="utf-8")

# Explicit final evaluation 2: EO+IR combined validation.
eval_eo_ir_yaml_path.write_text(f"""path: {DATASET_ROOT.as_posix()}
train: eo/images/train
val:
  - eo/images/val
  - ir/images/val

{class_block()}""", encoding="utf-8")

experiment_config_path = REPO_DIR / EXPERIMENT_CONFIG
experiment_config_path.parent.mkdir(parents=True, exist_ok=True)
experiment_config_path.write_text(f"""id: E01
name: source_rgb_yolo11s
method: source_rgb
model: yolo11s-seg.pt
dataset: {DATASET_YAML}
train_modality: RGB
test_modality: IR
augmentation: none
epochs: 100
imgsz: 640
batch: auto
seed: 42
""", encoding="utf-8")

print("Training YAML:\n", dataset_yaml_path.read_text())
print("IR eval YAML:\n", eval_ir_yaml_path.read_text())
print("EO+IR eval YAML:\n", eval_eo_ir_yaml_path.read_text())
print("Experiment config:\n", experiment_config_path.read_text())


## Stage 3 - Install Dependencies And Check T4x2

In [ ]:
stage("Stage 3 - Install dependencies")
run_live([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)

stage("Stage 3 - GPU check")
import torch
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("device_count:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
if torch.cuda.device_count() < 2:
    raise RuntimeError("Expected Kaggle T4x2. Enable GPU T4x2 before training.")


## Stage 4 - Run Smoke Or Full Experiment With Live Logs

In [ ]:
stage("Stage 4 - Training")
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")

resume_args = []
if YOLO_RESUME == "auto":
    resume_args = ["--resume-if-available"]
elif YOLO_RESUME.lower() in {"1", "true", "yes"}:
    resume_args = ["--resume"]

train_command = [
    sys.executable, "-m", "domain_adaptation_segmentation.training.run_experiment",
    "--config", EXPERIMENT_CONFIG,
    "--output-root", str(OUTPUT_ROOT),
    "--device", YOLO_DEVICE,
    "--epochs", YOLO_EPOCHS,
    "--batch", YOLO_BATCH,
    "--workers", YOLO_WORKERS,
    "--patience", YOLO_PATIENCE,
    *resume_args,
]

print("Command:", " ".join(train_command))
run_live(train_command, cwd=REPO_DIR, env=env)

collect_command = [
    sys.executable, "-m", "domain_adaptation_segmentation.training.collect_results",
    "--runs-root", str(OUTPUT_ROOT),
    "--output-dir", str(REPORT_DIR),
]
run_live(collect_command, cwd=REPO_DIR, env=env)

# Post-training evaluation uses best.pt on two evaluation sets:
#   1. IR-only: the main domain-transfer metric.
#   2. EO+IR: combined validation to show overall cross-domain behavior.
best_model = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME / "ultralytics" / "train" / "weights" / "best.pt"
eval_root = OUTPUT_ROOT / "evaluations"
if not best_model.exists():
    raise FileNotFoundError(best_model)

for eval_name, eval_yaml in [
    ("eval_ir", EVAL_IR_YAML),
    ("eval_eo_ir", EVAL_EO_IR_YAML),
]:
    eval_command = [
        sys.executable, "-m", "domain_adaptation_segmentation.training.evaluate_model",
        "--model", str(best_model),
        "--data", eval_yaml,
        "--output-root", str(eval_root),
        "--name", eval_name,
        "--device", YOLO_EVAL_DEVICE,
        "--imgsz", "640",
        "--batch", YOLO_BATCH,
        "--workers", YOLO_WORKERS,
    ]
    run_live(eval_command, cwd=REPO_DIR, env=env)

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
show_tail(run_dir / "stdout.log", lines=80)


## Stage 5 - Print Results In Notebook

In [ ]:
stage("Stage 5 - Results")
import pandas as pd

run_dir = OUTPUT_ROOT / "experiments" / EXPERIMENT_NAME
status_path = run_dir / "status.json"
results_path = run_dir / "results.csv"
summary_path = REPO_DIR / REPORT_DIR / "summary_results.csv"

print("run_dir:", run_dir)
if status_path.exists():
    status = json.loads(status_path.read_text(encoding="utf-8"))
    print(json.dumps(status, indent=2)[:4000])
else:
    print("status.json missing")

if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = [column.strip() for column in df.columns]
    display(df.tail())
    last = df.iloc[-1]
    print("\nLatest key metrics:")
    for col in ["epoch", "metrics/mAP50(M)", "metrics/mAP50-95(M)", "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]:
        if col in df.columns:
            print(f"{col}: {last[col]}")
else:
    print("results.csv missing")

if summary_path.exists():
    print("\nSummary table:")
    display(pd.read_csv(summary_path))


eval_rows = []
for eval_name in ["eval_ir", "eval_eo_ir"]:
    metrics_path = OUTPUT_ROOT / "evaluations" / eval_name / "metrics.json"
    print(f"\nEvaluation: {eval_name}")
    if not metrics_path.exists():
        print("missing", metrics_path)
        continue
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    metrics = payload.get("metrics", {})
    row = {"eval": eval_name}
    for key in [
        "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)",
        "metrics/precision(M)", "metrics/recall(M)", "metrics/mAP50(M)", "metrics/mAP50-95(M)",
    ]:
        row[key] = metrics.get(key)
    eval_rows.append(row)

if eval_rows:
    eval_df = pd.DataFrame(eval_rows)
    print("\nPost-training best.pt evaluations:")
    display(eval_df)


## Stage 6 - Store And Download Results

In [ ]:
stage("Stage 6 - Package outputs")
bundle_dir = WORK_DIR / f"{RUN_STAGE}_e01_artifacts"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

if OUTPUT_ROOT.exists():
    shutil.copytree(OUTPUT_ROOT, bundle_dir / OUTPUT_ROOT.name)
if (REPO_DIR / REPORT_DIR).exists():
    shutil.copytree(REPO_DIR / REPORT_DIR, bundle_dir / "tables")

archive_base = WORK_DIR / f"{RUN_STAGE}_e01_results"
archive_path = shutil.make_archive(str(archive_base), "zip", bundle_dir)
print("Result zip:", archive_path)

if Path("/kaggle").exists():
    print("Kaggle cannot silently download files to your local system from the kernel.")
    print("Use this FileLink or the right-side Output/Files panel to download:")
    from IPython.display import FileLink, display
    display(FileLink(archive_path))
elif running_on_colab():
    from google.colab import files
    print("Starting browser download via google.colab.files.download...")
    files.download(archive_path)
else:
    print("Download the result zip from this path:")
    print(archive_path)
